In [1]:
import os
import random
from glob import glob
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.cluster import KMeans

# ---------------------------
# CONFIGURATION
# ---------------------------
class Config:
    # --- PATHS ---
    # Ensure these point to the folders containing the data
    TRAIN_ROOT = "/kaggle/input/second-dataset-1" 
    TEST_ROOT = "/kaggle/input/second-dataset-test/SECOND_total_test/test" 
    
    # --- PIPELINE SETTINGS ---
    # Threshold: If KMeans detects < 0.5% of pixels as change, we skip classification
    CHANGE_PIXEL_THRESHOLD = 0.005 
    
    # --- HYPERPARAMETERS ---
    EPOCHS = 20
    BATCH_SIZE = 4 
    LR = 1e-4
    CHANGE_CLASS_WEIGHT = 5.0 
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 0 

cfg = Config()

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ---------------------------
# DATASET UTILS & CLASSES
# ---------------------------
COLOR_TO_CLASS = {
    (255,255,255): 0, (0,0,255): 1, (128,128,128): 2,
    (0,128,0): 3, (0,255,0): 4, (128,0,0): 5,
}
NUM_BASE_CLASSES = len(set(COLOR_TO_CLASS.values()))
NUM_CHANGE_CLASSES = 1 + NUM_BASE_CLASSES * NUM_BASE_CLASSES

def rgb_to_index_mask(pil_mask):
    arr = np.array(pil_mask.convert("RGB"), dtype=np.uint8)
    H,W,_ = arr.shape
    out = np.zeros((H,W), dtype=np.uint8)
    out.fill(2) # Default background
    for rgb, cls in COLOR_TO_CLASS.items():
        match = np.all(arr == np.array(rgb, dtype=np.uint8), axis=2)
        out[match] = cls
    return out

def make_change_map(label1_idx, label2_idx):
    C = NUM_BASE_CLASSES
    out = np.zeros_like(label1_idx, dtype=np.int64)
    same = (label1_idx == label2_idx)
    out[same] = 0
    diff = ~same
    if diff.sum() > 0:
        code = (label1_idx * C + label2_idx) + 1
        out[diff] = code[diff]
    return out

class SECONDChangeDataset(Dataset):
    def __init__(self, base_dir, which="train", transform=None, valid_ids=None):
        # --- PATH AUTO-CORRECTION LOGIC ---
        if which == "train":
            # Check specific structure first, fallback to root
            p = os.path.join(base_dir, "SECOND_train_set")
            base = p if os.path.isdir(p) else base_dir
        elif which == "test":
            # Check deep structure first
            p1 = os.path.join(base_dir, "SECOND_total_test", "test")
            # Check intermediate structure
            p2 = os.path.join(base_dir, "test")
            
            if os.path.isdir(p1): base = p1
            elif os.path.isdir(p2): base = p2
            else: base = base_dir # Fallback to root
        
        self.im1_dir = os.path.join(base, "im1")
        self.im2_dir = os.path.join(base, "im2")
        self.l1_dir = os.path.join(base, "label1")
        self.l2_dir = os.path.join(base, "label2")
        
        # Check if valid
        if not os.path.isdir(self.im1_dir):
            print(f"WARNING: Could not find 'im1' directory in {base}")
        
        all_ids = sorted([os.path.splitext(os.path.basename(p))[0] for p in glob(os.path.join(self.im1_dir, "*.png"))])
        
        # FILTERING: strictly use valid_ids if provided
        if valid_ids is not None:
            self.ids = [pid for pid in all_ids if pid in valid_ids]
        else:
            self.ids = all_ids
            
        self.transform = transform or (T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]))
        self.base = base

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid = self.ids[idx]
        p1 = os.path.join(self.im1_dir, pid + ".png")
        p2 = os.path.join(self.im2_dir, pid + ".png")
        l1 = os.path.join(self.l1_dir, pid + ".png")
        l2 = os.path.join(self.l2_dir, pid + ".png")
        
        im1_t = self.transform(Image.open(p1).convert("RGB"))
        im2_t = self.transform(Image.open(p2).convert("RGB"))
        
        # Load masks if they exist (test set might not have them in some competitions)
        if os.path.exists(l1) and os.path.exists(l2):
            lab1_idx = rgb_to_index_mask(Image.open(l1))
            lab2_idx = rgb_to_index_mask(Image.open(l2))
            change_target = make_change_map(lab1_idx, lab2_idx)
        else:
            # Placeholder for inference if labels missing
            change_target = np.zeros((im1_t.shape[1], im1_t.shape[2]), dtype=np.int64)
            
        return im1_t, im2_t, torch.from_numpy(change_target).long(), pid

# ---------------------------
# STAGE 1: UNSUPERVISED CLUSTERING (VGG)
# ---------------------------
class VGGFeatExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.VGG19_BN_Weights.DEFAULT if pretrained else None
        vgg = models.vgg19_bn(weights=weights)
        self.features = nn.Sequential(*list(vgg.features.children())[:27]) # Up to Block 3
        
    def forward(self, x):
        return self.features(x)

def run_clustering_check(im1_tensor, im2_tensor, model, device):
    """Returns True if clustering detects significant change, else False."""
    with torch.no_grad():
        f1 = model(im1_tensor.unsqueeze(0).to(device))
        f2 = model(im2_tensor.unsqueeze(0).to(device))
        
    diff = torch.abs(f2 - f1)[0]
    diff_up = F.interpolate(diff.unsqueeze(0), size=(256,256), mode='bilinear', align_corners=False)[0].cpu().numpy()
    
    C,H,W = diff_up.shape
    flat = diff_up.reshape(C, -1).T
    
    # Subsample for speed
    n = flat.shape[0]
    sample_idx = np.random.choice(n, 5000, replace=False) if n > 5000 else np.arange(n)
    sample = flat[sample_idx]

    kmeans = KMeans(n_clusters=2, random_state=0, n_init=1).fit(sample) 
    labels = kmeans.predict(flat)
    
    means = [np.linalg.norm(flat[labels==cl], axis=1).mean() if np.any(labels==cl) else 0 for cl in [0,1]]
    changed_cl = np.argmax(means)
    
    change_ratio = np.sum(labels == changed_cl) / n
    return change_ratio > cfg.CHANGE_PIXEL_THRESHOLD

def filter_dataset_with_clustering(dataset, device):
    """Runs VGG-KMeans on the entire dataset. Returns valid IDs."""
    print(f"--- Pipeline Stage 1: Clustering Proposal ({len(dataset)} images) ---")
    
    # --- FIX: Handle Empty Dataset Safely ---
    if len(dataset) == 0:
        print("Warning: Dataset is empty. Skipping clustering check.")
        return []
    
    model = VGGFeatExtractor(pretrained=True).to(device).eval()
    valid_ids = []
    
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    for im1, im2, _, pid in tqdm(loader, desc="Clustering Check"):
        is_changed = run_clustering_check(im1[0], im2[0], model, device)
        if is_changed:
            valid_ids.append(pid[0])
            
    print(f"Filter Complete: Kept {len(valid_ids)}/{len(dataset)} images ({len(valid_ids)/len(dataset):.1%})")
    return valid_ids

# ---------------------------
# STAGE 2: CLASSIFICATION MODEL
# ---------------------------
class VGG19Segmentation(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = models.VGG19_BN_Weights.DEFAULT if pretrained else None
        vgg = models.vgg19_bn(weights=weights)
        features = list(vgg.features.children())
        
        features[0] = nn.Conv2d(6, 64, kernel_size=3, padding=1)
        # Encoder Slicing
        pool_indices = [i for i, x in enumerate(features) if isinstance(x, nn.MaxPool2d)]
        self.enc1 = nn.Sequential(*features[0:pool_indices[0]])
        self.pool1 = features[pool_indices[0]]
        self.enc2 = nn.Sequential(*features[pool_indices[0]+1:pool_indices[1]])
        self.pool2 = features[pool_indices[1]]
        self.enc3 = nn.Sequential(*features[pool_indices[1]+1:pool_indices[2]])
        self.pool3 = features[pool_indices[2]]
        self.enc4 = nn.Sequential(*features[pool_indices[2]+1:pool_indices[3]])
        self.pool4 = features[pool_indices[3]]
        self.enc5 = nn.Sequential(*features[pool_indices[3]+1:pool_indices[4]])
        self.pool5 = features[pool_indices[4]]

        self.center = self._dec_block(512, 512)
        self.dec5 = self._dec_block(1024, 256)
        self.dec4 = self._dec_block(768, 128)
        self.dec3 = self._dec_block(384, 64)
        self.dec2 = self._dec_block(192, 64)
        self.dec1 = self._dec_block(128, 64)
        self.final = nn.Conv2d(64, num_classes, kernel_size=1)

    def _dec_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True),
            nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        e5 = self.enc5(self.pool4(e4))
        c = self.center(self.pool5(e5))
        d5 = self.dec5(torch.cat([F.interpolate(c, size=e5.shape[2:], mode='bilinear'), e5], 1))
        d4 = self.dec4(torch.cat([F.interpolate(d5, size=e4.shape[2:], mode='bilinear'), e4], 1))
        d3 = self.dec3(torch.cat([F.interpolate(d4, size=e3.shape[2:], mode='bilinear'), e3], 1))
        d2 = self.dec2(torch.cat([F.interpolate(d3, size=e2.shape[2:], mode='bilinear'), e2], 1))
        d1 = self.dec1(torch.cat([F.interpolate(d2, size=e1.shape[2:], mode='bilinear'), e1], 1))
        return F.interpolate(self.final(d1), size=x.shape[2:], mode='bilinear')

# ---------------------------
# TRAINING & EVALUATION
# ---------------------------
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_acc = 0.0, 0.0
    w = torch.ones(NUM_CHANGE_CLASSES).to(device)
    w[1:] = cfg.CHANGE_CLASS_WEIGHT
    
    for im1, im2, target, _ in tqdm(loader, desc="Train"):
        im1, im2, target = im1.to(device), im2.to(device), target.to(device)
        logits = model(torch.cat([im1, im2], 1))
        loss = F.cross_entropy(logits, target, weight=w)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        
        total_loss += loss.item() * im1.size(0)
        total_acc += (logits.argmax(1) == target).float().mean().item() * im1.size(0)
    return total_loss/len(loader.dataset), total_acc/len(loader.dataset)

def evaluate_pipeline_performance(model, raw_dataset, valid_ids_set, device):
    """
    Evalutes the FULL PIPELINE on the raw dataset.
    - If ID in valid_ids_set: Uses Model Prediction
    - If ID NOT in valid_ids_set: Uses "No Change" (0) Prediction
    """
    # Safety Check for Empty Dataset
    if len(raw_dataset) == 0:
        print("Warning: Raw test dataset is empty. Cannot evaluate.")
        return 0.0, 0.0, 0.0

    model.eval()
    intersection = torch.zeros(2).to(device)
    union = torch.zeros(2).to(device)
    loader = DataLoader(raw_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    print(">>> Calculating Binary mIoU for Full Pipeline...")
    with torch.no_grad():
        for im1, im2, target, pid in tqdm(loader, desc="Full Eval"):
            pid = pid[0]
            target = target.to(device)
            bin_target = (target > 0).long()
            
            # PIPELINE LOGIC
            if pid in valid_ids_set:
                # Passed filter: Run VGG
                im1, im2 = im1.to(device), im2.to(device)
                logits = model(torch.cat([im1, im2], 1))
                preds = logits.argmax(1)
                bin_preds = (preds > 0).long()
            else:
                # Blocked by filter: Predict No Change (All Zeros)
                bin_preds = torch.zeros_like(bin_target)

            # Update Metrics
            for cls in [0, 1]:
                inter = ((bin_preds == cls) & (bin_target == cls)).sum()
                area_pred = (bin_preds == cls).sum()
                area_target = (bin_target == cls).sum()
                u = area_pred + area_target - inter
                intersection[cls] += inter
                union[cls] += u

    iou_nc = intersection[0] / (union[0] + 1e-6)
    iou_c = intersection[1] / (union[1] + 1e-6)
    binary_miou = (iou_nc + iou_c) / 2.0
    
    return binary_miou.item() * 100, iou_nc.item() * 100, iou_c.item() * 100

# ---------------------------
# MAIN EXECUTION
# ---------------------------
if __name__ == "__main__":
    print(f">>> STARTING UNIFIED PIPELINE ON {cfg.DEVICE}")
    
    # 1. INITIALIZE RAW DATASETS
    raw_train_ds = SECONDChangeDataset(cfg.TRAIN_ROOT, which="train")
    # This will now warn you if paths are wrong, instead of crashing later
    raw_test_ds  = SECONDChangeDataset(cfg.TEST_ROOT, which="test")
    
    # 2. PIPELINE STAGE 1: CLUSTERING FILTER
    # Note: We need the SET of valid IDs for fast lookup during evaluation
    train_ids = filter_dataset_with_clustering(raw_train_ds, cfg.DEVICE)
    test_ids_list = filter_dataset_with_clustering(raw_test_ds, cfg.DEVICE)
    test_ids_set = set(test_ids_list) # Convert to set for O(1) lookup
    
    # 3. PIPELINE STAGE 2: CLASSIFICATION TRAINING
    # Only load the images that passed the filter
    if len(train_ids) > 0:
        train_ds = SECONDChangeDataset(cfg.TRAIN_ROOT, which="train", valid_ids=train_ids)
        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS)
        
        print(f"\n[Stage 2] Training VGG19 on {len(train_ds)} filtered samples...")
        model = VGG19Segmentation(num_classes=NUM_CHANGE_CLASSES).to(cfg.DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR)
        
        # Train Loop
        for epoch in range(cfg.EPOCHS):
            loss, acc = train_epoch(model, train_loader, optimizer, cfg.DEVICE)
            print(f"Epoch {epoch+1}: Loss {loss:.4f} | Acc {acc:.4f}")
            
        torch.save(model.state_dict(), "best_pipeline_model.pt")
        print("--> Model Saved")
        
        # 4. FINAL EVALUATION (FULL PIPELINE)
        # We pass the RAW test dataset and the set of IDs allowed to run the model
        if len(raw_test_ds) > 0:
            bin_miou, iou_nc, iou_c = evaluate_pipeline_performance(model, raw_test_ds, test_ids_set, cfg.DEVICE)

            print("-" * 30)
            print(f"Final Pipeline Binary mIoU: {bin_miou:.2f}%")
            print("-" * 30)
            print(f"  - No Change IoU:          {iou_nc:.2f}%")
            print(f"  - Change IoU:             {iou_c:.2f}%")
            print("-" * 30)
        else:
            print("Skipping Evaluation: Test dataset is empty.")
        
    else:
        print("Error: No images passed the clustering filter. Check threshold.")

>>> STARTING UNIFIED PIPELINE ON cuda
--- Pipeline Stage 1: Clustering Proposal (2968 images) ---


Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth
100%|██████████| 548M/548M [00:02<00:00, 209MB/s]


Clustering Check:   0%|          | 0/2968 [00:00<?, ?it/s]

Filter Complete: Kept 2956/2968 images (99.6%)
--- Pipeline Stage 1: Clustering Proposal (1694 images) ---


Clustering Check:   0%|          | 0/1694 [00:00<?, ?it/s]

...

Epoch 19: Loss 0.6187 | Acc 0.8585


Train:   0%|          | 0/739 [00:00<?, ?it/s]

Epoch 20: Loss 0.5785 | Acc 0.8662
--> Model Saved
>>> Calculating Binary mIoU for Full Pipeline...


Full Eval:   0%|          | 0/1694 [00:00<?, ?it/s]

------------------------------
Final Pipeline Binary mIoU: 64.68%
------------------------------
  - No Change IoU:          82.05%
  - Change IoU:             47.31%
------------------------------
